# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do not treat as dict

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Let's review the available record sets, fields, and their `@id`s according to the Croissant specification.

Each record set, field, and column is uniquely identified by its `@id`. We'll iterate over record sets, display their `@id`, and show the available fields/columns for exploration.

In [ ]:
# List available record sets and their fields (all by @id)
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets defined in metadata (check Croissant schema)")
else:
    for rs in record_sets:
        print(f"RecordSet: @id={rs['@id']}\n  Name: {rs.get('name', 'N/A')}\n  Fields/Columns @id:")")
        for fld in rs.get('fields', []):
            print(f"    - {fld['@id']} ({fld.get('name', 'N/A')})")
        # If columns are used in addition to fields:
        for col in rs.get('columns', []):
            print(f"    - {col['@id']} (column)")

For demonstration, let's enumerate and sample available records for each record set by their `@id`.

In [ ]:
# Iterate and show some records for each record set by @id
record_set_ids = [rs['@id'] for rs in (metadata.record_sets or [])]
if not record_set_ids:
    print("No record sets found to preview.")
else:
    for rs_id in record_set_ids:
        print(f"\nSample record(s) from RecordSet @id: {rs_id}")
        for idx, record in enumerate(dataset.records(record_set=rs_id)):
            print(record)
            if idx >= 1:  # Only show first 2
                break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above.

In [ ]:
# Prepare to load all record sets into pandas DataFrames
dataframes = {}

if not record_set_ids:
    print("No record sets available for data extraction.")
else:
    for rs_id in record_set_ids:
        print(f"Loading record set {rs_id}...")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records; columns: {df.columns.tolist()}")

# For demonstration, pick the first record set (if any) to explore
if record_set_ids:
    primary_rs_id = record_set_ids[0]
    print(f"\nFields in main RecordSet (@id={primary_rs_id}):")
    print(dataframes[primary_rs_id].columns.tolist())
    display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll choose a numeric field by `@id` from the loaded DataFrame. If your dataset has a field like 'MW' (molecular weight), 'coverage', 'peptide_counts', etc., you may update `numeric_field_id` accordingly based on the prior data overview.

In [ ]:
# Example EDA
import numpy as np

# Choose the main record set and a numeric field
if record_set_ids:
    df = dataframes[primary_rs_id]

    # Auto-choose a likely numeric field if present
    possible_numeric_fields = [
        c for c in df.columns
        if any(s in c.lower() for s in ['mw', 'weight', 'coverage', 'count', 'abundance', 'intensity', 'score'])
    ]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        # Clean field for numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    else:
        print("No obvious numeric field found--please update 'numeric_field'.")
        numeric_field = None

    if numeric_field and df[numeric_field].dtype.kind in 'fi':
        # Filtering (e.g., MW > 20000)
        threshold = np.nanpercentile(df[numeric_field], 75)  # 75th percentile as dynamic threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f} (75th percentile)")

        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by a categorical field, e.g., 'sample' or 'accession' or similar
        possible_group_fields = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field} (first 5 groups):")
            print(grouped_df.head())
        else:
            print("No string field available for grouping.")
    else:
        print("No valid numeric field for EDA.")
else:
    print("No record set loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field's distribution and group averages
if record_set_ids and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping done above, plot group means
    if 'group_field' in locals():
        plt.figure(figsize=(10, 4))
        grouped_df.head(20).plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field} (first 20 groups)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("Nothing to visualize, no numeric field present.")

## 6. Conclusion
In this notebook, we:
- Loaded [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset via its Croissant schema with `mlcroissant`.
- Explored available record sets and their fields by `@id`.
- Loaded tabular data into pandas DataFrames using record set `@id`s.
- Applied basic EDA: filtering and normalization on a numeric field, with summary group statistics.
- Visualized numeric values and distributions.

You can extend this notebook by performing domain-specific analyses, selecting different `@id` fields for deeper insight, or integrating with modeling workflows using the extracted DataFrames.